# VLM Vehicle Attribute Identification — Final Findings

Zero-shot Gemini (`gemini-3.1-flash-lite`) vs fine-tuned, on held-out CompCars test
images (final sample: n=500, seed 42). Covers accuracy, confidence intervals,
top mismatches, and the full cost/token breakdown for every experiment run
(inference evals + the two Vertex AI fine-tuning jobs).

**To add a new run**, add one entry to `RUNS` in the config cell — nothing
else needs to change.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.width", 120)
RESULTS_DIR = Path("../../results")  # notebooks/vlm/ -> repo root -> results/

## 1. Runs included

Final, report-grade results are the **n=500** runs. The n=100 tuned-v1 run is
kept for reference only — it's the "naive" fine-tuning attempt (3 epochs,
small adapter) that showed no real improvement, which is why a second,
properly configured run (v2, 10 epochs, larger adapter) was done.

In [ ]:
RUNS = [
    {
        "label": "Zero-shot (raw prompt)",
        "path": RESULTS_DIR / "vlm_eval_compcars_classification_raw_n500.csv",
        "attrs": ["make", "model", "body_type"],
        "n_target": 500,
    },
    {
        "label": "Zero-shot (constrained prompt)",
        "path": RESULTS_DIR / "vlm_eval_compcars_classification_constrained_n500.csv",
        "attrs": ["make", "model", "body_type"],
        "n_target": 500,
    },
    {
        "label": "Zero-shot — colour",
        "path": RESULTS_DIR / "vlm_eval_compcars_surveillance_raw_n500.csv",
        "attrs": ["colour"],
        "n_target": 500,
    },
    {
        "label": "Fine-tuned v2 (10 epochs, adapter 4)",
        "path": RESULTS_DIR / "vlm_eval_compcars_classification_tuned_v2_n500.csv",
        "attrs": ["make", "model", "body_type"],
        "n_target": 500,
    },
    {
        "label": "Fine-tuned v1 (3 epochs, adapter 2) — reference only, n=100",
        "path": RESULTS_DIR / "vlm_eval_compcars_classification_tuned.csv",
        "attrs": ["make", "model", "body_type"],
        "n_target": 100,
    },
]

# Training cost for the two Vertex AI fine-tuning jobs (not in the eval CSVs --
# those only cover inference calls). Figures from the job status checks.
TUNING_JOBS = [
    {"label": "Tuning job v1 (3 epochs, adapter 2)", "tokens": 587_983 * 3, "cost_usd": 5.29},
    {"label": "Tuning job v2 (10 epochs, adapter 4)", "tokens": 587_983 * 10, "cost_usd": 17.60},
]

## 2. Load and score

Accuracy and 95% CI (Wilson score interval, more reliable than the normal
approximation at these sample sizes) come straight from the `_ok` columns the
eval script already computed — no re-scoring here, so this always matches
what `eval_vlm_zero_shot.py` printed when the run finished.

In [ ]:
def wilson_ci(successes: int, n: int, z: float = 1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = successes / n
    denom = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n)
    return ((centre - margin) / denom, (centre + margin) / denom)


def load_run(run: dict):
    df = pd.read_csv(run["path"])
    if "error" in df.columns:
        df["error"] = df["error"].fillna("")  # pandas reads "" back as NaN
    n_total = len(df)
    n_failed = (df["error"].astype(str) != "").sum() if "error" in df else 0
    rows = []
    for attr in run["attrs"]:
        ok_col, gt_col = f"{attr}_ok", f"{attr}_gt"
        if ok_col not in df.columns:
            continue
        scored = df[df[gt_col].notna()] if gt_col in df.columns else df
        scored = scored[scored["error"].astype(str) == ""] if "error" in scored else scored
        n = len(scored)
        successes = int(scored[ok_col].astype(bool).sum())
        acc = successes / n if n else np.nan
        lo, hi = wilson_ci(successes, n)
        rows.append({
            "run": run["label"], "attribute": attr, "n": n,
            "accuracy": acc, "ci_low": lo, "ci_high": hi,
            "n_total_images": n_total, "n_failed": n_failed,
        })
    tokens_in = df["prompt_tokens"].sum() if "prompt_tokens" in df else 0
    tokens_out = df["output_tokens"].sum() if "output_tokens" in df else 0
    return pd.DataFrame(rows), int(tokens_in), int(tokens_out)


# gemini-3.1-flash-lite pricing (USD per token). Tuned-model inference is
# billed at 1.5x the base rate for Gemini-3-generation models.
PRICE_IN, PRICE_OUT = 0.25 / 1_000_000, 1.50 / 1_000_000
TUNED_PRICE_IN, TUNED_PRICE_OUT = PRICE_IN * 1.5, PRICE_OUT * 1.5

summary_rows, cost_rows = [], []
for run in RUNS:
    if not run["path"].exists():
        print(f"  [missing] {run['path']} -- skipping '{run['label']}'")
        continue
    scored_df, tin, tout = load_run(run)
    summary_rows.append(scored_df)
    is_tuned = "Fine-tuned" in run["label"]
    pin, pout = (TUNED_PRICE_IN, TUNED_PRICE_OUT) if is_tuned else (PRICE_IN, PRICE_OUT)
    cost_rows.append({
        "run": run["label"], "tokens_in": tin, "tokens_out": tout,
        "est_cost_usd": round(tin * pin + tout * pout, 4),
        "kind": "inference",
    })

summary = pd.concat(summary_rows, ignore_index=True) if summary_rows else pd.DataFrame()
costs = pd.DataFrame(cost_rows)
summary

## 3. Accuracy by run, with 95% CI

In [ ]:
display_tbl = summary.copy()
for c in ["accuracy", "ci_low", "ci_high"]:
    display_tbl[c] = (display_tbl[c] * 100).round(1)
display_tbl[["run", "attribute", "n", "accuracy", "ci_low", "ci_high", "n_failed"]]

## 4. Headline comparison chart

n=500 runs only (the n=100 v1 reference run is excluded here — it's not on
the same sample size, so bars wouldn't be a fair visual comparison).

In [ ]:
main_runs = [r["label"] for r in RUNS if r["n_target"] == 500]
chart_data = summary[summary["run"].isin(main_runs)]
attrs_present = [a for a in ["make", "model", "body_type", "colour"] if a in chart_data["attribute"].unique()]

fig, axes = plt.subplots(1, len(attrs_present), figsize=(5.5 * len(attrs_present), 5), squeeze=False)
axes = axes[0]

for ax, attr in zip(axes, attrs_present):
    sub = chart_data[chart_data["attribute"] == attr]
    x = np.arange(len(sub))
    acc = sub["accuracy"].to_numpy() * 100
    err_low = acc - sub["ci_low"].to_numpy() * 100
    err_high = sub["ci_high"].to_numpy() * 100 - acc
    colors = ["#4C72B0" if "Fine-tuned" not in r else "#C44E52" for r in sub["run"]]
    ax.bar(x, acc, yerr=[err_low, err_high], capsize=4, color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels([r.split(" (")[0] for r in sub["run"]], rotation=30, ha="right", fontsize=8)
    ax.set_ylim(0, 100)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"{attr}  (n={int(sub['n'].iloc[0])})")
    for xi, a in zip(x, acc):
        ax.text(xi, a + 2, f"{a:.1f}%", ha="center", fontsize=9)

fig.suptitle("Zero-shot vs fine-tuned VLM accuracy (n=500, 95% CI)")
fig.tight_layout()
plt.show()

## 5. Top confusions per run

What the model actually gets wrong, not just how often.

In [ ]:
def top_mismatches(run: dict, attr: str, k: int = 8) -> pd.DataFrame:
    df = pd.read_csv(run["path"])
    if "error" in df.columns:
        df["error"] = df["error"].fillna("")
    gt_col, pred_col, ok_col = f"{attr}_gt", f"{attr}_pred", f"{attr}_ok"
    if ok_col not in df.columns:
        return pd.DataFrame()
    wrong = df[(df["error"].astype(str) == "") & (~df[ok_col].astype(bool)) & df[gt_col].notna()]
    if wrong.empty:
        return pd.DataFrame()
    return (
        wrong.groupby([gt_col, pred_col], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(k)
    )


for run in RUNS:
    if not run["path"].exists() or run["n_target"] != 500:
        continue  # skip the n=100 reference run here, same reasoning as the chart
    for attr in run["attrs"]:
        mism = top_mismatches(run, attr)
        if mism.empty:
            continue
        print(f"--- {run['label']} -- {attr} ---")
        display(mism)

## 6. Cost & tokens

Two different kinds of spend: **inference** (every eval run shown above, at
Gemini API rates) and **training** (the two Vertex AI fine-tuning jobs — not
in the eval CSVs, since the jobs themselves aren't scored, just billed).

In [ ]:
print("--- Inference (evaluation calls) ---")
display(costs)
print(f"Total inference spend: ${costs['est_cost_usd'].sum():.4f}")

print("\n--- Training (fine-tuning jobs) ---")
tuning_df = pd.DataFrame(TUNING_JOBS)
display(tuning_df)
print(f"Total training spend: ${tuning_df['cost_usd'].sum():.2f}")

total = costs["est_cost_usd"].sum() + tuning_df["cost_usd"].sum()
print(f"\n=== Total project spend so far: ${total:.2f} ===")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
labels = list(costs["run"]) + list(tuning_df["label"])
values = list(costs["est_cost_usd"]) + list(tuning_df["cost_usd"])
colors = ["#4C72B0"] * len(costs) + ["#DD8452"] * len(tuning_df)
y = np.arange(len(labels))
ax.barh(y, values, color=colors)
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("Cost (USD)")
ax.set_title("Cost by run (blue = inference, orange = training)")
for yi, v in zip(y, values):
    ax.text(v, yi, f" ${v:.2f}" if v >= 1 else f" ${v:.3f}", va="center", fontsize=8)
fig.tight_layout()
plt.show()

## Key findings

- **Make**: ~91% zero-shot, 92.8% fine-tuned (v2) -- a small, directionally
  positive gain that sits within the ~±3pt CI at n=500 (not a strong claim on
  its own).
- **Model (exact name)**: ~55% zero-shot (either prompt) vs **66.4%
  fine-tuned (v2)** -- an ~11-point gain that exceeds the CI margin. This is
  the headline result: fine-tuning produced a real, defensible improvement
  specifically on the hardest sub-task.
- **Body type**: constraining the zero-shot prompt (listing the exact
  CompCars labels) already recovered most of the achievable gain (81.2% ->
  85.5%); fine-tuning (85.7%) did not improve meaningfully beyond that.
- **Colour**: 89.2% zero-shot; not covered by fine-tuning, since the training
  manifest (web-nature) has no colour labels -- only the surveillance
  manifest does, and it lacks model/body_type.
- **The first fine-tuning attempt (v1: 3 epochs, small adapter) showed no
  measurable improvement** -- the gains above came only after diagnosing
  under-training and re-running with 10 epochs and a larger LoRA adapter
  (v2). This is itself a useful methodological finding: default
  auto-selected tuning settings were not sufficient for this dataset size.
- **Total project spend**: inference evals + both training jobs, see the
  cost table above -- all well within a typical capstone budget.

### Caveats
- CIs are per-attribute Wilson intervals; a few points of difference between
  two runs may not be meaningful even where this notebook doesn't flag it --
  always check the CI columns in Section 3, not just the bar heights.
- The v1 fine-tuned run (n=100) is kept for reference only and is not
  plotted in the headline chart, since it isn't on the same sample size as
  everything else.
- Runs are only comparable when scored on the same image sample (same
  manifest, split, `-n`, and `--seed` in `eval_vlm_zero_shot.py`).